# 08 · El modelo facial real, a través del servicio

Se ejecuta en **Google Colab** (GPU T4), de arriba abajo. Los notebooks 02 y 06 midieron el reconocimiento con funciones de `ml/`.
Aquí se mide **por el camino del servicio**: el adaptador real (`app/infrastructure/facenet.py`, MTCNN + FaceNet) conectado a los casos de uso
`RegistrarRostro`, `IdentificarRostro` y `RetirarConsentimiento`, con las reglas I1, I2, I3 e I5 y los umbrales 0,70 y 0,50.

Qué comprueba:
1. **Paridad:** el adaptador del servicio da los mismos vectores que el pipeline de `ml/` (si no, lo medido antes no valdría para el servicio).
2. **Rendimiento por zonas:** propios (sesión dejada fuera), compañero no registrado y extraños de LFW; se compara con el notebook 06.
3. **El ciclo de vida de I1:** registrar → reconocer → retirar el consentimiento → ya no se reconoce y las fotos se borran.
4. **Latencia** de una identificación.

Antes de subir: genera `servicio.zip` (carpetas `app/` y `ml/` del repo `ia`) y ten a mano `luis.zip` y `aldana.zip`.
Las fotos y los vectores no se suben a git (I6); este notebook solo guarda métricas.

In [ ]:
import os, sys, zipfile
from google.colab import files

subidos = files.upload()      # servicio.zip + un zip por persona (luis.zip, aldana.zip...)
os.makedirs("/content/condohub-ia", exist_ok=True)
propios_dir = "/content/condohub-ml/datos/propios"
os.makedirs(propios_dir, exist_ok=True)
for nombre in subidos:
    z = zipfile.ZipFile(nombre)
    if nombre.lower().startswith("servicio"):
        z.extractall("/content/condohub-ia")
        continue
    persona = os.path.splitext(nombre)[0].split(" (")[0].lower()
    con_raiz = all(n.startswith(persona + "/") for n in z.namelist())
    z.extractall(propios_dir if con_raiz else os.path.join(propios_dir, persona))
os.environ["ML_DATA_DIR"] = "/content/condohub-ml"
os.chdir("/content/condohub-ia")
sys.path.insert(0, "/content/condohub-ia")
print("carpeta:", sorted(os.listdir("/content/condohub-ia")), "| personas:", sorted(os.listdir(propios_dir)))

In [ ]:
!pip install -q facenet-pytorch --no-deps
!pip install -q scikit-learn pyyaml pillow
import torch
print("GPU disponible:", torch.cuda.is_available())

## 1 · El adaptador real y las fotos del equipo

In [ ]:
import hashlib, io, json, time, yaml, numpy as np
from collections import Counter
from datetime import UTC, datetime, timedelta
from uuid import UUID, uuid4
from PIL import Image

from app.application.rostros import IdentificarRostro, RegistrarRostro, RetirarConsentimiento
from app.domain.roles import Rol, Solicitante
from app.domain.rostros import ReconocedorFacial, Umbrales
from app.infrastructure.composicion import RelojSistema
from app.infrastructure.facenet import ModeloFacialFacenet
from app.infrastructure.memoria import (AlmacenFotosMemoria, RepositorioDesconocidosMemoria,
                                        RepositorioPerfilesMemoria, RepositorioPrediccionesMemoria)
from ml.src import propios as fotos_propias
from ml.src.paths import dir_datos, dir_experimento

class ModeloConCache:
    """Calcula cada foto una sola vez: la evaluación repite el registro muchas veces con las mismas fotos."""
    def __init__(self, modelo):
        self._m, self._cache = modelo, {}
        self.nombre, self.version = modelo.nombre, modelo.version
    def embeber(self, imagen):
        clave = hashlib.sha1(imagen).hexdigest()
        if clave not in self._cache:
            self._cache[clave] = self._m.embeber(imagen)
        return self._cache[clave]

modelo_real = ModeloFacialFacenet()
modelo = ModeloConCache(modelo_real)

raiz = dir_datos() / "datos/propios"
fotos = fotos_propias.listar_fotos(raiz)               # I1: exige consentimiento.json de cada persona
personas = sorted({f.persona for f in fotos})
fotos_de = {}                                           # (persona, sesión) -> bytes de cada foto
for f in fotos:
    fotos_de.setdefault((f.persona, f.sesion), []).append(f.ruta.read_bytes())
consentimientos = {p: fotos_propias.leer_consentimiento(raiz / p) for p in personas}
print({p: {s: len(b) for (q, s), b in fotos_de.items() if q == p} for p in personas})

## 2 · Paridad con el pipeline de `ml/`

Los vectores del servicio (bytes de la foto → adaptador) frente a los de `ml/src/embeddings.py` (el de los notebooks 02, 03 y 06): la similitud coseno entre
los dos debe ser ≈ 1. Si no, hay una diferencia de preprocesado y las métricas de antes no valen para el servicio.

In [ ]:
from ml.src import embeddings as ml_emb

rng = np.random.default_rng(0)
muestra = [fotos[i] for i in rng.choice(len(fotos), size=min(30, len(fotos)), replace=False)]
embeber_ml = ml_emb.crear_embebedor()
vec_ml, detectado_ml = embeber_ml([fotos_propias.cargar_imagen(f.ruta) for f in muestra])

paridad, discrepancias = [], 0
for i, f in enumerate(muestra):
    v = modelo.embeber(f.ruta.read_bytes())
    if (v is None) != (not detectado_ml[i]):
        discrepancias += 1                       # uno detecta rostro y el otro no
    elif v is not None:
        paridad.append(float(np.dot(np.array(v), vec_ml[i])))
paridad = np.array(paridad)
print(f"fotos comparadas: {len(paridad)} | mínimo {paridad.min():.5f} | mediana {np.median(paridad):.5f} | discrepancias de detección: {discrepancias}")
assert paridad.min() > 0.99, "El servicio y ml/ no producen el mismo vector: revisar el preprocesado"

## 3 · El servicio armado con dobles en memoria y umbrales de I2

In [ ]:
UMBRAL_ALTO, UMBRAL_BAJO = 0.70, 0.50
FECHA_CONSENTIMIENTO = datetime.now(UTC) - timedelta(days=1)       # el texto del consentimiento es borrador: solo se prueba el servicio

def servicio(registro):
    """Servicio nuevo con `registro` = {persona: [bytes de foto]} ya registrado por RegistrarRostro (I1)."""
    perfiles, almacen = RepositorioPerfilesMemoria(), AlmacenFotosMemoria()
    desconocidos, predicciones, reloj = RepositorioDesconocidosMemoria(), RepositorioPrediccionesMemoria(), RelojSistema()
    registrar = RegistrarRostro(modelo, perfiles, almacen, reloj)
    identificar = IdentificarRostro(modelo, ReconocedorFacial(Umbrales(UMBRAL_ALTO, UMBRAL_BAJO)), perfiles,
                                    desconocidos, predicciones, reloj, retencion_desconocidos=timedelta(days=30))
    retirar = RetirarConsentimiento(perfiles, almacen, reloj, plazo=timedelta(hours=24))
    ids = {}
    for persona, bytes_fotos in registro.items():
        ids[persona] = uuid4()
        registrar.ejecutar(ids[persona], f"sub-{persona}", UUID(int=1), FECHA_CONSENTIMIENTO,
                           consentimientos[persona]["version"], bytes_fotos)
    return dict(identificar=identificar, retirar=retirar, ids=ids, perfiles=perfiles, almacen=almacen,
                desconocidos=desconocidos, predicciones=predicciones)

def fotos_completas(persona):
    return [b for (p, s), fs in fotos_de.items() if p == persona for b in fs]

def clasificar(resultado, esperado):
    """Etiqueta cómo salió una identificación frente a quién era de verdad (`esperado`: id o None)."""
    if resultado.resultado == "SIN_ROSTRO":
        return "sin_rostro"
    if resultado.resultado == "RECONOCIDO":
        return "reconocido_correcto" if esperado is not None and resultado.residente_id == esperado else "reconocido_otra_persona"
    if resultado.resultado == "DUDOSO":
        return "dudoso_candidato_correcto" if esperado is not None and resultado.residente_id == esperado else "dudoso_candidato_otro"
    return "desconocido"

## 4 · Propios: se deja una sesión fuera y se identifica con el servicio

In [ ]:
conteo_propios, por_sesion, latencias = Counter(), [], []
for persona in personas:
    sesiones = sorted({s for (p, s) in fotos_de if p == persona})
    for sesion in sesiones:
        resto = [b for s in sesiones if s != sesion for b in fotos_de[(persona, s)]]
        if not resto:
            continue
        registro = {q: fotos_completas(q) for q in personas if q != persona}
        registro[persona] = resto
        s = servicio(registro)
        c = Counter()
        for foto in fotos_de[(persona, sesion)]:
            t0 = time.perf_counter()
            r = s["identificar"].ejecutar(foto, camara="prueba")
            latencias.append((time.perf_counter() - t0) * 1000)
            c[clasificar(r, s["ids"][persona])] += 1
        conteo_propios.update(c)
        por_sesion.append({"persona": persona, "sesion": sesion, "fotos": sum(c.values()), **c})

total = sum(conteo_propios.values())
print(f"Propios ({total} fotos, sesión dejada fuera):")
for clave, n in conteo_propios.most_common():
    print(f"   {clave:<28} {n:>4}  ({100 * n / total:5.1f} %)")
for fila in por_sesion:
    print(fila)

## 5 · Compañero no registrado y extraños de LFW

In [ ]:
conteo_companeros = Counter()
for persona in personas:
    s = servicio({q: fotos_completas(q) for q in personas if q != persona})     # esta persona NO está registrada
    for foto in fotos_completas(persona):
        conteo_companeros[clasificar(s["identificar"].ejecutar(foto), None)] += 1
tot_c = sum(conteo_companeros.values())
print(f"Compañero NO registrado ({tot_c} fotos):", {k: f"{100 * v / tot_c:.1f} %" for k, v in conteo_companeros.items()})

In [ ]:
from ml.src import lfw

img_lfw, et_lfw, _ = lfw.cargar_lfw(10)
rng = np.random.default_rng(42)
elegidas = np.concatenate([rng.permutation(np.flatnonzero(et_lfw == p))[:2] for p in np.unique(et_lfw)])
bytes_extranos = []
for i in elegidas:
    buffer = io.BytesIO()
    Image.fromarray(img_lfw[i]).save(buffer, "JPEG", quality=95)
    bytes_extranos.append(buffer.getvalue())
del img_lfw
print(len(bytes_extranos), "fotos de extraños")

In [ ]:
s = servicio({q: fotos_completas(q) for q in personas})
conteo_extranos = Counter(clasificar(s["identificar"].ejecutar(b), None) for b in bytes_extranos)
tot_e = sum(conteo_extranos.values())
print(f"Extraños de LFW ({tot_e} fotos):", {k: f"{100 * v / tot_e:.1f} %" for k, v in conteo_extranos.items()})
print(f"I3: {len(s['desconocidos'].guardados)} desconocidos guardados | I5: {len(s['predicciones'].guardadas)} predicciones registradas")

## 6 · El ciclo de vida de I1: registrar → reconocer → retirar el consentimiento

In [ ]:
persona = personas[0]
s = servicio({persona: fotos_completas(persona)})
foto = fotos_completas(persona)[0]
antes = s["identificar"].ejecutar(foto)
fotos_antes = len(s["almacen"].objetos)
retiro = s["retirar"].ejecutar(Solicitante(f"sub-{persona}", frozenset({Rol.RESIDENTE})), s["ids"][persona])
despues = s["identificar"].ejecutar(foto)
perfil = s["perfiles"].por_residente(s["ids"][persona])
print(f"antes del retiro : {antes.resultado} (confianza {antes.confianza:.3f})")
print(f"fotos guardadas  : {fotos_antes} -> {len(s['almacen'].objetos)} | embeddings del perfil: {len(perfil.embeddings)}")
print(f"después del retiro: {despues.resultado} | plazo de borrado: {retiro.borrado_antes_de:%Y-%m-%d %H:%M} UTC")
assert antes.resultado == "RECONOCIDO" and despues.resultado == "DESCONOCIDO" and not s["almacen"].objetos

## 7 · Comparar con el notebook 06 y guardar

In [ ]:
from google.colab import files

previo = json.load(open("ml/experiments/2026-10-15_registro_equipo/metrics.json", encoding="utf-8"))["zonas"]["regla_inicial"]
pct = lambda c, k: 100 * c.get(k, 0) / max(1, sum(c.values()))
print(f"{'':<42}{'servicio':>10}{'notebook 06':>13}")
print(f"{'Propios reconocidos (correcto)':<42}{pct(conteo_propios, 'reconocido_correcto'):>9.1f}%{100 * previo['genuinos']['reconocido_correcto']:>12.1f}%")
print(f"{'Propios reconocidos como OTRA persona':<42}{pct(conteo_propios, 'reconocido_otra_persona'):>9.1f}%{100 * previo['genuinos']['reconocido_otra_persona']:>12.1f}%")
print(f"{'Compañero no registrado -> RECONOCIDO':<42}{pct(conteo_companeros, 'reconocido_otra_persona'):>9.1f}%{100 * previo['companeros_no_registrados']['RECONOCIDO']:>12.1f}%")
print(f"{'Extraños de LFW -> DESCONOCIDO':<42}{pct(conteo_extranos, 'desconocido'):>9.1f}%{100 * previo['extranos_prueba']['DESCONOCIDO']:>12.1f}%")
lat = np.array(latencias)
print(f"latencia por identificación: mediana {np.median(lat):.0f} ms | p95 {np.percentile(lat, 95):.0f} ms | máx {lat.max():.0f} ms ({'GPU' if torch.cuda.is_available() else 'CPU'})")

destino = dir_experimento("servicio_facial")
resultado = {
    "umbrales": {"alto": UMBRAL_ALTO, "bajo": UMBRAL_BAJO},
    "paridad_con_ml": {"fotos": int(len(paridad)), "minimo": float(paridad.min()), "mediana": float(np.median(paridad)),
                       "discrepancias_de_deteccion": discrepancias},
    "propios": dict(conteo_propios), "propios_por_sesion": por_sesion,
    "companeros_no_registrados": dict(conteo_companeros), "extranos_lfw": dict(conteo_extranos),
    "latencia_ms": {"mediana": float(np.median(lat)), "p95": float(np.percentile(lat, 95)), "dispositivo": "GPU" if torch.cuda.is_available() else "CPU"},
    "ciclo_I1": {"antes": antes.resultado, "despues": despues.resultado, "fotos_borradas": fotos_antes},
    "personas": {p: {"consentimiento": consentimientos[p]} for p in personas},
    "modelo": {"nombre": modelo.nombre, "version": modelo.version},
}
(destino / "metrics.json").write_text(json.dumps(resultado, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
files.download(str(destino / "metrics.json"))